# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



## Setup: Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"] = "WARNING"

In [1]:
from gai.lib.constants import DEFAULT_GUID
from gai.asm.agents import ToolUseAgent2
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.messages import FileMonologue
from gai.messages import FileDialogue, MessagePydantic

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

def print_chunk(chunk):
    """
    Helper function to print the chunk of data received from the agent.
    """
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
        else:
            if isinstance(chunk, list):
                for item in chunk:
                    if item.get("name"):
                        print(f'Tool: "{item["name"]}"')
                    if item.get("input"):
                        inputs = item.get("input")
                        if isinstance(inputs, dict):
                            for key, value in inputs.items():
                                if isinstance(value, str):
                                    if len(value) > 100:
                                        print(
                                            f"\tInput: {key} = {value[:100]}... (truncated)"
                                        )
                                    else:
                                        print(f"\tInput: {key} = {value}")

---

## Scenario 1: Context inferred from dialogue context

In this scenario, we create a hypothetical conversation where the user talks about weather in Singapore before asking for the time.

This is to demonstrate that the agent can infer that the user is asking for current time in Singapore.

If this doesn't work, the agent will ask the user to clarify the location.


In [2]:
# Reset the monologue to start fresh

monologue.reset()

# Create an artificial dialogue history for testing
messages = [
    MessagePydantic(
        **{
            "id": "b1e5f98c-f6eb-47de-a6e2-387510d970f9",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1751308157.270983,
                "order": 0,
            },
            "body": {
                "type": "chat.send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 0,
                "role": "user",
                "content": "It is a very nice weather in Singapore right now.",
            },
        }
    ),
    MessagePydantic(
        **{
            "id": "abbc7961-45dc-4973-aaf4-a6224ed35d37",
            "header": {
                "sender": "Sara",
                "recipient": "User",
                "timestamp": 1751308167.3488164,
                "order": 1,
            },
            "body": {
                "type": "chat.reply",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 1,
                "chunk_no": 10,
                "chunk": "<eom>",
                "role": "assistant",
                "content": "Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?",
            },
        }
    ),
]
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages, file_path=file_path)

agent = ToolUseAgent2(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue
)

user_message = "When is the next public holiday?"
recap = dialogue.extract_recap()

# Update dialogue

dialogue.add_user_message(recipient="Sara", content=user_message)

# INIT
print(f"\ncompleted state: {agent.fsm.state}")

# INIT -> HAS_MESSAGE
resp=await agent.start_async(user_message=user_message, recap=recap)
print(f"\ncompleted state: {agent.fsm.state}")

# HAS_MESSAGE -> CHAT
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)   
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)
print(f"\ncompleted state: {agent.fsm.state}")

# CHAT -> IS_TOOL_CALL
await agent.resume_async()
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["predicate_result"] is True
print(f"\ncompleted state: {agent.fsm.state}")



completed state: INIT

completed state: HAS_MESSAGE
I'll help you find information about the next public holiday. Let me search for current public holiday information.
Tool: "search"
	Input: search_query = next public holiday 2024 2025 upcoming

completed state: CHAT

completed state: IS_TOOL_CALL


In [3]:
# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)
print(f"\ncompleted state: {agent.fsm.state}")



Let me get the current date to determine which upcoming holiday is next.
Tool: "current_time"
	Input: format = YYYY-MM-DD

completed state: TOOL_USE


### c) Show monologue

See the monologue for details at `tmp/monologue.json`

In [5]:
import json
from gai.messages import message_helper

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size=message_helper.get_messages_length(agent.fsm.monologue.list_chat_messages())
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "68434fa0-fac4-476d-aad6-1b14551969fe",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1753672385.3054812,
        "order": 0
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 2,
        "content_type": "text",
        "role": "user",
        "content": "\n            Your name is ToolUseAgent within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            \n            \n## Goal\n\nWhen is the next public holiday?\n\n## Instructions\n\n- Find the best way to achieve the goal.\n\n\n            Here is a recap of the conversation:\n            User: It is a very nice weather in Singapore 

---

## Scenario 2: Agent interrupt user for input

In this scenario, we will not use dialogue recap.

We just ask the agent directly about the time without giving any context. The agent should ask the user for input to continue the conversation.

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input



In [2]:
monologue.reset()

file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
aggregated_client = McpAggregatedClient(["mcp-pseudo", "mcp-time", "mcp-web"])
agent = ToolUseAgent2(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

user_message = "When is the next public holiday? Please ask if you need more information."

# INIT
print(f"\ncompleted state: {agent.fsm.state}")

# INIT -> IS_TOOL_CALL
resp = await agent.start_async()
assert agent.fsm.state == "IS_TOOL_CALL"
print(f"\ncompleted state: {agent.fsm.state}")
print(f"\nis_tool_call_result: {agent.fsm.state_bag['is_tool_call_result']}")
print(f"\npredicate_result: {agent.fsm.state_bag['predicate_result']}")



completed state: INIT

completed state: IS_TOOL_CALL

is_tool_call_result: False

predicate_result: False


In [3]:
# IS_TOOL_CALL -> CHAT
resp = await agent.resume_async(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "CHAT"

Hello! I'm ToolUseAgent. To help you find the next public holiday, I need some additional information from you.
Tool: "user_input"

completed state: CHAT


In [4]:
# CHAT -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"
assert agent.fsm.state_bag["is_terminate_result"] is False
assert agent.fsm.state_bag["predicate_result"] is False


completed state: IS_TERMINATE


### d) This will keep asking for input until user provides input

In [5]:
# IS_TERMINATE -> IS_TOOL_CALL
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["is_tool_call_result"] is True
assert agent.fsm.state_bag["predicate_result"] is True
assert agent.fsm.state_bag["is_user_input"] is True

# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "TOOL_USE"

# TOOL_USE -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"


completed state: IS_TOOL_CALL

completed state: TOOL_USE

completed state: IS_TERMINATE


### e) Can resume normally after user input

In [6]:
# IS_TERMINATE -> IS_TOOL_CALL
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["is_tool_call_result"] is True
assert agent.fsm.state_bag["predicate_result"] is True
assert agent.fsm.state_bag["is_user_input"] is True

# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async("Use SGT")
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "TOOL_USE"



completed state: IS_TOOL_CALL
Thank you! I'll search for the next public holiday in Singapore (SGT timezone).
Tool: "search"
	Input: search_query = Singapore public holidays 2024 2025 next upcoming holiday dates

completed state: TOOL_USE


In [7]:
agent.monologue.list_messages()

[MessagePydantic(id='083275c3-a7f6-40ab-99fa-64dd0ddbc71d', header=MessageHeaderPydantic(sender='User', recipient='ToolUseAgent', timestamp=1753704666.5636134, order=0), body=MonologueBodyPydantic(type='monologue', state_name='CHAT', step_no=2, content_type='text', role='user', content='\n            Your name is ToolUseAgent within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            When is the next public holiday? Please ask if you need more information.\n            ')),
 MessagePydantic(id='69a23e85-2942-4e24-a0d3-a3328a080d15', header=MessageHeaderPydantic(sender='ToolUseAgent', recipient='User', timestamp=1753704671.2017133, order=1), body=MonologueBodyPydantic(type='monologue', state_name='CHAT', step_no=2, content_type='text', role='assistant', content=[

In [ ]:
# TOOL_USE -> HAS_MESSAGE
messages=agent.monologue.list_messages()
import json
for msg in messages:
    print(json.dumps(msg.model_dump(), indent=4))

{
    "id": "b248a868-5646-4b98-acc8-f0a179de4a18",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1753681258.5827053,
        "order": 0
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 2,
        "content_type": "text",
        "role": "user",
        "content": "\n            Your name is ToolUseAgent within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            When is the next public holiday? Please ask if you need more information.\n            "
    }
}
{
    "id": "b9a45c57-71e4-47c5-a2d4-21259e11de1c",
    "header": {
        "sender": "ToolUseAgent",
        "recipient": "User",
        "timestamp": 1753681262.809255,
        "order": 1
    },
    "body"

---

## Scenario 3: User interrupt agent with adhoc input

In this scenario, the user tries to distract the agent by interrupting it with an adhoc input.

### a) start


In [6]:
monologue.reset()

file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(file_path=file_path)

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

goal = "When is the next public holiday?"
user_message=f"""
## Goal

{goal}


"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

I'll help you find information about the next public holiday. Let me search for current public holiday information.
Tool: "search"
	Input: search_query = next public holiday 2024 2025 upcoming
I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.
Tool: "current_time"
	Input: format = YYYY-MM-DD


MessagePydantic(id='0932bf45-fe49-45cf-a668-32e0349d3263', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753272166.5349991, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.235', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.'))

### b) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [7]:
user_message = "Tell me a one paragraph joke."
resp = agent.interrupt(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue when user interrupt agent
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, sees nothing, and turns back to find the librarian has vanished. He stands there confused until he hears a faint voice from behind the counter saying, "Sorry, I was just getting your books - turns out we keep all the paranoia books in the back section because we're worried people might be watching them too closely. Also, I should mention that everyone who checks out these books gets followed home by our security team, but don't worry, it's just to make sure you're reading them properly and not sharing the secrets with anyone suspicious."

Now, getting back to your original question about the next public holiday - let me get the current date and provide you with accurate information about upcoming public holidays.
Thinking...


MessagePydantic(id='a5bea499-05d1-4a49-b6ed-02671553f906', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753272208.706365, order=5), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=2, step_no=1, message_id='00000000-0000-0000-0000-000000000000.342', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='Here\'s a one paragraph joke for you:\n\nA man walks into a library and asks for books on paranoia. The librarian whispers, "They\'re right behind you!" The man spins around frantically, sees nothing, and turns back to find the librarian has vanished. He stands there confused until he hears a faint voice from behind the counter saying, "Sorry, I was just getting your books - turns out we keep all the paranoia books in the back section because we\'re worried people might be watching them too closely. Also, I should mention that everyone who checks out these books gets followed home b

### d) Show dialogue

In [8]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: Sara, When is the next public holiday?
Sara: I notice you've repeated the same question. To find the next public holiday for you, I still need to know which country or region you're asking about, since public holidays are different in each location. 

Could you please tell me:
- Which country are you in?
- Or which country's public holidays are you interested in?

For example, are you asking about holidays in the United States, United Kingdom, Canada, Australia, or another country?
User: Sara, When is the next public holiday?
Sara: I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.
User: Sara, Tell me a one paragraph joke.
Sara: Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, sees nothing, and 